# Wykład 10 — Dynamiczne tablice. Jak zorganizować dane w pamięci?

Macierz z dwoma wierszami i trzema kolumnami ma sześć elementów. Czy żeby zapisać ją w pamięci, potrzebujemy jednego bloku, dwóch czy trzech? Wszystko zależy od tego, jak chcemy docierać do elementów.

Zaczniemy od jednego bloku liczb. Zbudujemy dwa sposoby przechowywania macierzy o wymiarach ustalanych podczas działania programu. Po drodze prześledzimy typy wskaźników i ustalimy, kto odpowiada za każde `free`.

**Po wykładzie** potrafisz zbudować obie reprezentacje, uzyskać dostęp do `A[i][j]`, przekazać dane do funkcji i zwolnić każdy przydzielony blok także po błędzie.

## W10.1 — Kto pamięta adres?

Zmienna wskaźnikowa przechowuje adres. Lokalny wskaźnik może zniknąć po wyjściu z funkcji, a blok przydzielony przez `malloc` nadal istnieje. Jak przekazać jego adres dalej, żeby móc z niego skorzystać i go zwolnić?

Funkcje przydzielania i zwalniania pamięci deklaruje nagłówek `<stdlib.h>`.

~~~c
void *malloc(size_t size);
void free(void *ptr);
~~~

`malloc` przyjmuje liczbę **bajtów**. Zwraca adres początku przydzielonego bloku albo `NULL`, gdy przydział się nie uda. W C możemy przypisać ten adres do wskaźnika odpowiedniego typu bez rzutowania. Przed odczytem każdego elementu wpisujemy w niego wartość.

![Blok przydzielony w funkcji pozostaje dostępny przez q po zakończeniu funkcji. Wywołanie free(q) kończy życie bloku, a zmienna q nadal istnieje.](figures/w10-lifetime.png)

Po udanym przydziale pilnujemy, by **raz zwolnić podstawę bloku**. Po `free(q)` blok przestaje istnieć, a wszystkie wskaźniki do niego przestają nadawać się do użycia. Przypisanie `q = NULL` nie zmienia innych zmiennych wskaźnikowych. `free(NULL)` nic nie robi.

## W10.2 — Ile elementów potrzebujemy?

Gdy podczas działania programu poznajemy liczbę pomiarów, możemy dobrać pojemność do danych. Policzmy liczbę bajtów dla `char`, `int` i `double`. Co się zmieni, gdy wybierzemy inny typ, a co pozostanie takie samo w pętli?

| Deklaracja lokalna | Skąd rozmiar? | Kiedy kończy się życie danych? |
|---|---|---|
| `int a[6];` | Stała znana przy kompilacji | Przy wyjściu z bloku |
| `int a[n];` | Zmienna długość (VLA), ustalana w chwili deklaracji | Przy wyjściu z zakresu deklaracji |
| `int *a = malloc(n * sizeof *a);` | Liczba elementów znana przed przydziałem | Przy `free(a)` |

VLA wprowadzono w C99. W C11 obsługa VLA zależy od implementacji. W tym kursie używamy `malloc` do przydziałów o rozmiarze ustalanym podczas wykonania. Dwie pierwsze deklaracje w tabeli dotyczą obiektów automatycznych. Terminy *stos* i *sterta* opisują częsty sposób implementacji.

`sizeof` jest operatorem. Dla `sizeof *p` poniżej kompilator bierze rozmiar typu elementu i nie odczytuje wartości wskazywanej przez `p`. Wynik ma typ `size_t` i wypisujemy go przez `%zu`. Typowe rozmiary `char`, `int` i `double` na Linux x86-64 to 1, 4 i 8 bajtów C. Tylko `sizeof(char) == 1` jest tu gwarantowane przez C.

### Najpierw znaki i liczby całkowite

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    enum { N = 3 };
    char *letters = malloc(N * sizeof *letters);
    if (letters == NULL)
        return EXIT_FAILURE;

    int *numbers = malloc(N * sizeof *numbers);
    if (numbers == NULL) {
        free(letters);
        return EXIT_FAILURE;
    }

    const char sample[N] = {'C', '1', '0'};
    for (size_t i = 0; i < N; ++i) {
        letters[i] = sample[i];
        numbers[i] = 11 + (int)i;
    }

    for (size_t i = 0; i < N; ++i)
        printf("%c %d\n", letters[i], numbers[i]);

    printf("sizeof(char) = %zu, sizeof(int) = %zu\n",
           sizeof *letters, sizeof *numbers);

    free(numbers);
    free(letters);
    return EXIT_SUCCESS;
}

**Interpretacja.** Pętle indeksują oba bloki tak samo. `%c` wypisuje pojedynczy znak, a `%d` wartość typu `int`.

### Liczba pomiarów od użytkownika

W pierwszej linii komórki zmień `5` na inną liczbę od 1 do 1000. Kernel C przekazuje tę liczbę jako argument programu. W terminalu po kompilacji podaj ją po nazwie programu, np. `./measurements 5`.

In [ ]:
//% args: 5
#include <errno.h>
#include <inttypes.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>

/* Sprawdzamy cały zapis liczby przed konwersją do size_t. */
static size_t read_size(const char *text, size_t limit)
{
    if (*text < '0' || *text > '9')
        return 0;

    char *end;
    errno = 0;
    uintmax_t value = strtoumax(text, &end, 10);
    if (errno == ERANGE || *end != '\0' || value == 0 || value > limit)
        return 0;
    return (size_t)value;
}

int main(int argc, char *argv[])
{
    size_t n = argc == 2 ? read_size(argv[1], 1000) : 0;
    double *samples;
    if (n == 0 || n > SIZE_MAX / sizeof *samples) {
        fputs("Podaj liczbę całkowitą od 1 do 1000.\n", stderr);
        return EXIT_FAILURE;
    }

    size_t bytes = n * sizeof *samples;
    samples = malloc(bytes);
    if (samples == NULL) {
        fputs("Nie udało się przydzielić pamięci.\n", stderr);
        return EXIT_FAILURE;
    }

    for (size_t i = 0; i < n; ++i)
        samples[i] = 0.5 * i;
    for (size_t i = 0; i < n; ++i)
        printf("%.1f%c", samples[i], i + 1 == n ? '\n' : ' ');
    printf("Liczba bajtów = %zu\n", bytes);

    free(samples);
    return EXIT_SUCCESS;
}

**Interpretacja.** Najpierw odczytujemy całą liczbę i odrzucamy wynik poza zakresem. Test z `SIZE_MAX` poprzedza mnożenie. Dla pięciu elementów otrzymujemy `0.0 0.5 1.0 1.5 2.0`. Ile bajtów zamówi program na Twojej platformie?

## W10.3 — Dwie współrzędne, jeden blok?

Każdy punkt ma współrzędne $x$ i $y$. Możemy rozdzielić je na dwa bloki albo podzielić jeden większy blok na dwie części. W obu przypadkach użyjemy `x[i]` i `y[i]`. Ile razy w każdym przypadku wywołamy `free`?

W [samples/data1.dat](samples/data1.dat) pierwsza liczba oznacza liczbę punktów, a za nią następują pary `x y` w kolejności punktów.

![Dwa przydziały x i y mają osobne podstawy. Przy wspólnym przydziale y = x + n wskazuje do wnętrza bloku zaczynającego się pod x.](figures/w10-coordinates.png)

Wspólny blok grupuje wszystkie wartości `x`, a potem wszystkie wartości `y`. Wyrażenie `y = x + n` przesuwa wskaźnik o $n$ elementów typu `double`. W obu układach pętla odczytu może pozostać identyczna.

In [ ]:
#include <ctype.h>
#include <errno.h>
#include <inttypes.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>

/* Wczytaj jeden token tekstowy. Odrzuć zbyt długi token w całości. */
static int read_token(FILE *file, char token[64])
{
    size_t length = 0;
    int ch;
    do {
        ch = fgetc(file);
    } while (ch != EOF && isspace(ch));

    while (ch != EOF && !isspace(ch)) {
        if (length == 63 || ch == '\0')
            return 0;
        token[length++] = (char)ch;
        ch = fgetc(file);
    }
    token[length] = '\0';
    return length > 0 && !ferror(file);
}
/* Sprawdzamy cały zapis liczby przed konwersją do size_t. */
static size_t read_size(const char *text, size_t limit)
{
    if (*text < '0' || *text > '9')
        return 0;

    char *end;
    errno = 0;
    uintmax_t value = strtoumax(text, &end, 10);
    if (errno == ERANGE || *end != '\0' || value == 0 || value > limit)
        return 0;
    return (size_t)value;
}

static int read_coordinate(FILE *file, double *value)
{
    char token[64], *end;
    if (!read_token(file, token))
        return 0;
    errno = 0;
    double result = strtod(token, &end);
    if (end == token || *end != '\0' || errno == ERANGE || !isfinite(result))
        return 0;
    *value = result;
    return 1;
}

static int show_points(int shared)
{
    FILE *file = fopen("samples/data1.dat", "r");
    if (file == NULL)
        return 0;

    char token[64];
    double *x = NULL, *y = NULL;
    int ok = 0;
    size_t n = read_token(file, token) ? read_size(token, 1000) : 0;
    if (n == 0 || n > SIZE_MAX / sizeof *x / 2)
        goto cleanup;

    x = malloc((shared ? 2 * n : n) * sizeof *x);
    if (x == NULL)
        goto cleanup;
    y = shared ? x + n : malloc(n * sizeof *y);
    if (y == NULL)
        goto cleanup;

    for (size_t i = 0; i < n; ++i) {
        if (!read_coordinate(file, &x[i]) || !read_coordinate(file, &y[i]))
            goto cleanup;
    }

    puts(shared ? "Jeden blok" : "Dwa bloki");
    for (size_t i = 0; i < n; ++i)
        printf("%.6g %.6g\n", x[i], y[i]);
    ok = 1;

cleanup:
    if (!shared)
        free(y);
    free(x);
    fclose(file);
    return ok;
}

int main(void)
{
    if (!show_points(0) || !show_points(1)) {
        fputs("Nie udało się wczytać punktów.\n", stderr);
        return EXIT_FAILURE;
    }
    return EXIT_SUCCESS;
}

**Interpretacja.** Obie wersje drukują te same trzy pary. Przy dwóch przydziałach zwalniamy `y` i `x`. Przy jednym przydziale zwalniamy `x`; `y` wskazuje do wnętrza tego samego bloku. Przy awarii zamykamy plik i zwalniamy każdy blok, który udało się przydzielić.

## W10.4 — Jak funkcja może zmienić mój wskaźnik?

Jeśli mamy adres zmiennej, możemy zmienić jej wartość. A jeśli tą zmienną jest wskaźnik? Porównamy zwracanie adresu z przekazywaniem adresu zmiennej wskaźnikowej.

### Dwa kroki od pp do liczby

Dla `int a = 5; int *p = &a; int **pp = &p;` zmienna `p` zawiera adres `a`, a zmienna `pp` zawiera adres `p`. Wyrażenie `*pp` oznacza `p`. Kolejne `*` prowadzi do `a`.

![Trzy osobne obiekty pp, p i a. Wartość &p prowadzi do p, a wartość &a do liczby 5.](figures/w10-pointer-chain.png)

In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 5;
    int *p = &a;
    int **pp = &p;

    printf("%d %d %d %d\n", a, *p, **pp, pp[0][0]);
    printf("p = %p, &a = %p\n", (void *)p, (void *)&a);
    printf("pp = %p, &p = %p\n", (void *)pp, (void *)&p);
    return 0;
}

**Interpretacja.** Wszystkie cztery odczyty dają 5. Indeks zero oznacza dereferencję bez przesunięcia. Tu istnieje tylko jeden obiekt na każdym z końców strzałek, więc inne indeksy nie są dostępne.

### Zwróć adres albo przekaż adres wskaźnika

C przekazuje argumenty **przez wartość**. Gdy przekażemy `x`, funkcja dostanie kopię adresu. Zmiana tej kopii nie zmieni `x` w funkcji wywołującej. Gdy przekażemy `&x`, kopia zawiera adres samej zmiennej `x`, więc przypisanie do `*out` zmieni `x`.

In [ ]:
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>

double *allocdouble(size_t n)
{
    if (n == 0 || n > SIZE_MAX / sizeof(double))
        return NULL;
    return malloc(n * sizeof(double));
}

void allocdouble2(size_t n, double **out)
{
    *out = allocdouble(n);
}

int main(void)
{
    size_t n = 3;
    double *left = allocdouble(n);
    if (left == NULL)
        return EXIT_FAILURE;

    double *right = NULL;
    allocdouble2(n, &right);
    if (right == NULL) {
        free(left);
        return EXIT_FAILURE;
    }

    for (size_t i = 0; i < n; ++i) {
        left[i] = (double)i;
        right[i] = left[i] + 0.5;
        printf("%.1f %.1f\n", left[i], right[i]);
    }

    free(right);
    free(left);
    return EXIT_SUCCESS;
}

**Umowa.** Obie wersje przekazują adres nowego bloku albo `NULL`. W wersji z parametrem wyjściowym `out` wskazuje zapisywalną zmienną, w której można zastąpić poprzednią wartość. Tutaj `right` zaczyna jako `NULL`. Po udanej alokacji wywołujący inicjalizuje dane i zwalnia blok.

### A jeśli adres prowadzi do zwykłej zmiennej?

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int a = 0, b = 6;
    int **links = malloc(2 * sizeof *links);
    if (links == NULL)
        return EXIT_FAILURE;

    links[0] = &a;
    links[1] = &b;
    printf("%d %d\n", *links[0], *links[1]);
    printf("%d %d\n", links[0][0], links[1][0]);

    free(links);  /* Zwalniamy tablicę adresów. */
    return EXIT_SUCCESS;
}

**Interpretacja.** Tablica adresów jest przydzielona dynamicznie. `a` i `b` to obiekty automatyczne. Zwalniamy blok `links`. Adresy zapisane w jego elementach prowadzą do obiektów, których życie kończy się przy wyjściu z `main`.

## W10.5 — Gdzie zaczyna się drugi wiersz?

W tablicy `int A[2][3]` drugi wiersz następuje po trzech liczbach pierwszego. Żeby znaleźć pozycję w płaskim bloku, pomijamy $i$ pełnych wierszy i idziemy jeszcze o $j$ elementów.

W układzie wierszowym (*row-major*) element o indeksach $0\le i<\mathrm{rows}$ i $0\le j<\mathrm{cols}$ ma pozycję liniową

$$
k = i \cdot \mathrm{cols} + j.
$$

![Macierz 2 na 3 i płaska kolejność sześciu wartości. Pozycja 5 odpowiada elementowi [1][2].](figures/w10-row-major.png)

Typ `A` to tablica dwóch wierszy typu `int [3]`. W typowym wyrażeniu `A` daje wskaźnik do pierwszego wiersza, typu `int (*)[3]`. Dodanie 1 do tego wskaźnika przesuwa o cały wiersz. W programie poniżej `p` jest osobnym płaskim blokiem, więc `p[k]` może przechodzić między logicznymi wierszami. Do elementów `A` docieramy przez `A[i][j]`.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    enum { ROWS = 2, COLS = 3 };
    int A[ROWS][COLS] = {{11, 12, 13}, {21, 22, 23}};
    int *p = malloc(ROWS * COLS * sizeof *p);
    if (p == NULL)
        return EXIT_FAILURE;

    for (size_t i = 0; i < ROWS; ++i)
        for (size_t j = 0; j < COLS; ++j)
            p[i * COLS + j] = A[i][j];

    for (size_t i = 0; i < ROWS; ++i) {
        printf("&A[%zu][0] = %p\n", i, (void *)&A[i][0]);
        for (size_t j = 0; j < COLS; ++j) {
            size_t k = i * COLS + j;
            printf("A[%zu][%zu] = %d, p[%zu] = %d\n",
                   i, j, A[i][j], k, p[k]);
        }
    }

    free(p);
    return EXIT_SUCCESS;
}

**Interpretacja.** Dla `i = 1` i `j = 2` mamy `k = 5`, a oba odczyty dają 23. Adresy są zależne od uruchomienia. Porównaj **wartości** i indeksy.

## W10.6 — Od jednego indeksu do dwóch

Najpierw ułożymy sześć liczb w bloku `p`. Potem nazwiemy początki dwóch wierszy, zbierzemy ich adresy w tablicy i dobierzemy liczbę miejsc na adresy podczas działania programu.

![Cztery sposoby dostępu do bloku 11, 12, 13, 21, 22, 23. p wskazuje początek bloku. p0 i p1, potem A[0] i A[1], wskazują początki wierszy. W ostatnim kroku także tablica adresów powstaje w malloc.](figures/w10-four-steps.png)

### Krok 1. Jeden blok typu int

Każda liczba ma pojedynczy indeks. Wyrażenie `p[3]` oznacza element, który traktujemy jako początek drugiego wiersza. Czy `p[3]` jest liczbą, czy adresem?

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int *p = malloc(6 * sizeof *p);
    if (p == NULL)
        return EXIT_FAILURE;

    p[0] = 11; p[1] = 12; p[2] = 13;
    p[3] = 21; p[4] = 22; p[5] = 23;

    printf("%d %d %d\n", p[0], p[1], p[2]);
    printf("%d %d %d\n", p[3], p[4], p[5]);

    free(p);
    return EXIT_SUCCESS;
}

### Krok 2. Dwa wskaźniki na początki wierszy

`p0 = p` i `p1 = p + 3` to adresy w tym samym przydziale. W `p1[2]` przesunięcie o dwa elementy zaczyna się od początku drugiego wiersza.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int *p = malloc(6 * sizeof *p);
    if (p == NULL)
        return EXIT_FAILURE;

    p[0] = 11; p[1] = 12; p[2] = 13;
    p[3] = 21; p[4] = 22; p[5] = 23;

    int *p0 = p;
    int *p1 = p + 3;
    printf("%d %d %d\n", p0[0], p0[1], p0[2]);
    printf("%d %d %d\n", p1[0], p1[1], p1[2]);

    free(p);
    return EXIT_SUCCESS;
}

### Krok 3. Tablica dwóch wskaźników

`int *A[2]` deklaruje automatyczną tablicę dwóch elementów typu `int *`. Teraz `A[i]` wybiera adres początku wiersza, a drugi indeks wybiera element tego wiersza.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int *p = malloc(6 * sizeof *p);
    if (p == NULL)
        return EXIT_FAILURE;

    p[0] = 11; p[1] = 12; p[2] = 13;
    p[3] = 21; p[4] = 22; p[5] = 23;

    int *A[2] = {p, p + 3};
    printf("%d %d %d\n", A[0][0], A[0][1], A[0][2]);
    printf("%d %d %d\n", A[1][0], A[1][1], A[1][2]);

    free(p);
    return EXIT_SUCCESS;
}

### Krok 4. Dynamiczna tablica wskaźników

Zamiast deklarować dwa miejsca przydzielamy blok na `rows` adresów. Wynik trzymamy w `int **A`, wskaźniku do elementu typu `int *`. Tutaj `rows` wynosi 2, ale liczba miejsc w bloku może zależeć od danych.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int *p = malloc(6 * sizeof *p);
    if (p == NULL)
        return EXIT_FAILURE;

    p[0] = 11; p[1] = 12; p[2] = 13;
    p[3] = 21; p[4] = 22; p[5] = 23;

    size_t rows = 2;
    int **A = malloc(rows * sizeof *A);
    if (A == NULL) {
        free(p);
        return EXIT_FAILURE;
    }
    A[0] = p;
    A[1] = p + 3;
    printf("%d %d %d\n", A[0][0], A[0][1], A[0][2]);
    printf("%d %d %d\n", A[1][0], A[1][1], A[1][2]);

    free(A);
    free(p);
    return EXIT_SUCCESS;
}

Dopiero teraz rozpiszmy oba indeksy:

~~~c
A[i]       == *(A + i)          // int *
A[i][j]    == *(A[i] + j)       // int
A[i][j]    == *(*(A + i) + j)    // int
~~~

**Interpretacja.** Każdy krok daje te same dwa wiersze. W ostatnim kroku `free(A)` zwalnia tablicę adresów, a `free(p)` zwalnia dane. Składnia `A[i][j]` działa też przy tablicy wierszy z W10.5, lecz pierwszy indeks wybiera wtedy wiersz jako podtablicę. W obu przypadkach krok odpowiada typowi elementu, po którym przesuwa się wskaźnik.

## W10.7 — Czy wiersze muszą sąsiadować?

Każdy wiersz może mieć własny blok. Możemy też ułożyć wiersze w jednym bloku danych i zapisać ich adresy w drugim. Obie wersje pozwalają pisać `A[i][j]`. Co musi wiedzieć funkcja, która zwalnia macierz?

![Po lewej tablica adresów A i dwa niezależne bloki danych. Po prawej tablica A wskazuje początki wierszy we wspólnym bloku p. Pod rysunkami kolejność zwalniania.](figures/w10-layouts.png)

- **Osobne wiersze.** Na $r$ wierszy potrzeba $r+1$ bloków. Wiersze mogą leżeć w dowolnych miejscach i mogą mieć różne długości. Poniższy program przydziela tę samą liczbę elementów w każdym wierszu, by porównać prostokątne macierze. Po awarii w środku pętli zwalniamy tylko wiersze przydzielone do tej chwili.
- **Wspólny blok danych.** Wystarczą dwa przydziały. Adres `A[i] = p + i * cols` prowadzi do początku wiersza w bloku `p`. Można przejść liniowo przez wszystkie elementy tego bloku. Żeby go zwolnić, używamy jego podstawy.

### Porównaj oba wydruki

Pierwsza linia komórki ustala `rows` i `cols`. Zmień `2 3` na `3 2`. Program przyjmuje do 1000 elementów i sprawdza granice iloczynów przed obliczeniem liczby bajtów.

In [ ]:
//% args: 2 3
#include <errno.h>
#include <inttypes.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>

/* Sprawdzamy cały zapis liczby przed konwersją do size_t. */
static size_t read_size(const char *text, size_t limit)
{
    if (*text < '0' || *text > '9')
        return 0;

    char *end;
    errno = 0;
    uintmax_t value = strtoumax(text, &end, 10);
    if (errno == ERANGE || *end != '\0' || value == 0 || value > limit)
        return 0;
    return (size_t)value;
}
/* Wymiary prostokąta i iloczyny mieszczą się w limitach. */
static int valid_shape(size_t rows, size_t cols)
{
    return rows > 0 && cols > 0 &&
           rows <= SIZE_MAX / sizeof(int *) &&
           rows <= SIZE_MAX / sizeof(int) / cols &&
           rows <= 1000 / cols;
}

static int **alloc2d(size_t rows, size_t cols)
{
    if (!valid_shape(rows, cols))
        return NULL;

    int **A = malloc(rows * sizeof *A);
    if (A == NULL)
        return NULL;

    int *p = malloc(rows * cols * sizeof *p);
    if (p == NULL) {
        free(A);
        return NULL;
    }
    for (size_t i = 0; i < rows; ++i)
        A[i] = p + i * cols;
    return A;
}

static void free2d(int **A)
{
    if (A != NULL) {
        free(A[0]);
        free(A);
    }
}

static void print2d(int **A, size_t rows, size_t cols)
{
    for (size_t i = 0; i < rows; ++i) {
        for (size_t j = 0; j < cols; ++j)
            printf("%d%c", A[i][j], j + 1 == cols ? '\n' : ' ');
    }
}

static void free_rows(int **A, size_t count)
{
    for (size_t i = 0; i < count; ++i)
        free(A[i]);
    free(A);
}

static int **alloc_rows(size_t rows, size_t cols)
{
    if (!valid_shape(rows, cols))
        return NULL;

    int **A = malloc(rows * sizeof *A);
    if (A == NULL)
        return NULL;

    for (size_t i = 0; i < rows; ++i) {
        A[i] = malloc(cols * sizeof *A[i]);
        if (A[i] == NULL) {
            free_rows(A, i);
            return NULL;
        }
    }
    return A;
}

static void fill2d(int **A, size_t rows, size_t cols)
{
    for (size_t i = 0; i < rows; ++i)
        for (size_t j = 0; j < cols; ++j)
            A[i][j] = 1 + (int)(i * cols + j);
}

int main(int argc, char *argv[])
{
    size_t rows = argc == 3 ? read_size(argv[1], 1000) : 0;
    size_t cols = argc == 3 ? read_size(argv[2], 1000) : 0;
    if (!valid_shape(rows, cols)) {
        fputs("Podaj dodatnie wymiary o iloczynie najwyżej 1000.\n", stderr);
        return EXIT_FAILURE;
    }

    int **rows_owned = alloc_rows(rows, cols);
    if (rows_owned == NULL)
        return EXIT_FAILURE;

    int **shared = alloc2d(rows, cols);
    if (shared == NULL) {
        free_rows(rows_owned, rows);
        return EXIT_FAILURE;
    }

    fill2d(rows_owned, rows, cols);
    fill2d(shared, rows, cols);
    puts("Osobne wiersze");
    print2d(rows_owned, rows, cols);
    puts("Wspólny blok danych");
    print2d(shared, rows, cols);

    free2d(shared);
    free_rows(rows_owned, rows);
    return EXIT_SUCCESS;
}

**Interpretacja.** Te same wartości powstają z tej samej pętli i dają ten sam wynik dla obu sposobów alokacji. Tablica wskaźników w układzie ciągłym przechowuje podstawę danych w `A[0]`. Zachowaj ją do końca albo przechowuj osobno w `p`. Zwolnienie każdego `A[i]` we wspólnym bloku byłoby błędem.

**Pytanie.** Jeśli wiersze są oddzielne i mają długości 2 i 4, jaka dodatkowa informacja będzie potrzebna pętli odczytującej drugi wiersz?

## W10.8 — Jak opisać umowę z funkcją?

Drukowanie potrzebuje wymiarów i dostępu do elementów. Zwalnianie potrzebuje wiedzy o przydzielonych blokach. Przećwiczymy też skalowanie pomiarów do przedziału $[0,1]$ oraz kopiowanie wiersza i przekątnej macierzy prostokątnej.

### Kontrakt macierzy dla funkcji

| Funkcja z W10.7 | Co przekazujemy i za co odpowiada funkcja? |
|---|---|
| `alloc2d(rows, cols)` | Dodatnie wymiary, maksymalnie 1000 elementów. Zwraca adres tablicy wskaźników do jednego bloku danych albo `NULL`. Przy niepowodzeniu sama sprząta. Po sukcesie pamięć na liczby trzeba wypełnić. |
| `print2d(A, rows, cols)` | Poprawny prostokąt z zainicjalizowanymi elementami. Funkcja pożycza dostęp do danych. Działa z obiema reprezentacjami. |
| `free2d(A)` | Wynik `alloc2d` albo `NULL`. Po udanym przydziale `rows > 0`, a `A[0]` wciąż zawiera podstawę bloku danych. Funkcja zwalnia dane i tablicę adresów. |
| `free_rows(A, count)` | Tablica adresów z osobnymi blokami. `count` mówi, ile początkowych wierszy faktycznie przydzielono. |

### Przeskaluj pomiary

Funkcje 1D, takie jak `print(double *a, size_t n)`, potrzebują adresu pierwszego elementu i długości. Tak samo wywołamy je dla tablicy automatycznej i dynamicznego bloku. Zapis `double a[]` w parametrze funkcji oznacza tu ten sam typ co `double *a`.

Żeby najmniejsza wartość przeszła na 0, a największa na 1, stosujemy do **każdego** elementu wzór

$$
x' = \frac{x-\min}{\max-\min}.
$$

W tej funkcji zakładamy skończone pomiary o rozstępie mieszczącym się w typie `double`. Dla stałych danych wpisujemy same zera. Dla pustej tablicy kończymy przed jakimkolwiek odczytem.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

static void fill(double *a, size_t n)
{
    const double values[] = {4, -2, 10, 1, 7};
    for (size_t i = 0; i < n; ++i)
        a[i] = values[i % (sizeof values / sizeof values[0])];
}

static void print(double *a, size_t n)
{
    for (size_t i = 0; i < n; ++i)
        printf("%.2f%c", a[i], i + 1 == n ? '\n' : ' ');
}

static void normalize(double a[], size_t n)
{
    if (n == 0)
        return;

    double min = a[0], max = a[0];
    for (size_t i = 1; i < n; ++i) {
        if (a[i] < min)
            min = a[i];
        if (a[i] > max)
            max = a[i];
    }

    double range = max - min;
    for (size_t i = 0; i < n; ++i)
        a[i] = range == 0.0 ? 0.0 : (a[i] - min) / range;
}

int main(void)
{
    enum { N = 5 };
    double *data = malloc(N * sizeof *data);
    if (data == NULL)
        return EXIT_FAILURE;

    fill(data, N);
    print(data, N);
    normalize(data, N);
    print(data, N);
    free(data);

    double constant[] = {3, 3, 3};
    normalize(constant, 3);
    print(constant, 3);

    normalize(NULL, 0);
    return EXIT_SUCCESS;
}

**Interpretacja.** Pierwszy pomiar `4` także się zmienia, na `0.50`. Skrajne wartości `-2` i `10` przechodzą na `0.00` i `1.00`. To samo wywołanie funkcji działa dla automatycznej tablicy ze stałymi danymi.

### Skopiuj wiersz i przekątną, wstaw wiersz

Dla tablicy o znanej szerokości możemy zapisać parametr `int A[][3]`. Funkcja otrzymuje wskaźnik do wiersza i zna rozmiar kroku między wierszami. W wersji z parametrem `int **A` pierwszy indeks odczytuje adres wiersza zapisany w tablicy adresów. W obu przypadkach funkcja dostaje wymiary, których ma przestrzegać.

**Ćwiczenie.** Dla prostokąta 2×3 i 3×2 przewidź liczbę elementów kopiowanego wiersza i przekątnej. Poniższe rozwiązanie kopiuje wiersz o indeksie 1 i przekątną, a na końcu wstawia kopię tego wiersza z powrotem do wiersza 0. Bufory są oddzielone od pamięci macierzy. Bufor z kopią wiersza staje się źródłem dla wstawiania. Każda funkcja sprawdza pojemność bufora przed zapisem lub odczytem.

In [ ]:
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>

/* Wymiary prostokąta i iloczyny mieszczą się w limitach. */
static int valid_shape(size_t rows, size_t cols)
{
    return rows > 0 && cols > 0 &&
           rows <= SIZE_MAX / sizeof(int *) &&
           rows <= SIZE_MAX / sizeof(int) / cols &&
           rows <= 1000 / cols;
}

static int **alloc2d(size_t rows, size_t cols)
{
    if (!valid_shape(rows, cols))
        return NULL;

    int **A = malloc(rows * sizeof *A);
    if (A == NULL)
        return NULL;

    int *p = malloc(rows * cols * sizeof *p);
    if (p == NULL) {
        free(A);
        return NULL;
    }
    for (size_t i = 0; i < rows; ++i)
        A[i] = p + i * cols;
    return A;
}

static void free2d(int **A)
{
    if (A != NULL) {
        free(A[0]);
        free(A);
    }
}

static void print2d(int **A, size_t rows, size_t cols)
{
    for (size_t i = 0; i < rows; ++i) {
        for (size_t j = 0; j < cols; ++j)
            printf("%d%c", A[i][j], j + 1 == cols ? '\n' : ' ');
    }
}

static void print_fixed(int A[][3], size_t rows)
{
    for (size_t i = 0; i < rows; ++i)
        printf("%d %d %d\n", A[i][0], A[i][1], A[i][2]);
}

static int copyrow(int **A, size_t rows, size_t cols, size_t row,
                   int *buffer, size_t capacity)
{
    if (row >= rows || capacity < cols)
        return 0;
    for (size_t j = 0; j < cols; ++j)
        buffer[j] = A[row][j];
    return 1;
}

static int copydiag(int **A, size_t rows, size_t cols,
                    int *buffer, size_t capacity)
{
    size_t count = rows < cols ? rows : cols;
    if (capacity < count)
        return 0;
    for (size_t i = 0; i < count; ++i)
        buffer[i] = A[i][i];
    return 1;
}

static int insertrow(int **A, size_t rows, size_t cols, size_t row,
                     const int *buffer, size_t capacity)
{
    if (row >= rows || capacity < cols)
        return 0;
    for (size_t j = 0; j < cols; ++j)
        A[row][j] = buffer[j];
    return 1;
}

static int demonstrate(size_t rows, size_t cols)
{
    int **A = alloc2d(rows, cols);
    if (A == NULL)
        return 0;

    for (size_t i = 0; i < rows; ++i)
        for (size_t j = 0; j < cols; ++j)
            A[i][j] = 10 * (int)(i + 1) + (int)(j + 1);

    int row[3], diag[2];
    int ok = copyrow(A, rows, cols, 1, row, 3) &&
             copydiag(A, rows, cols, diag, 2) &&
             insertrow(A, rows, cols, 0, row, 3);

    if (ok) {
        printf("Kopia wiersza:");
        for (size_t j = 0; j < cols; ++j)
            printf(" %d", row[j]);
        printf("\nPrzekątna:");
        size_t count = rows < cols ? rows : cols;
        for (size_t i = 0; i < count; ++i)
            printf(" %d", diag[i]);
        putchar('\n');
        puts("Po wstawieniu");
        print2d(A, rows, cols);
    }

    /* Niepoprawne żądania muszą zakończyć się przed dostępem do bufora. */
    ok = ok && !copyrow(A, rows, cols, rows, row, 3) &&
         !copyrow(A, rows, cols, 0, row, cols - 1) &&
         !copydiag(A, rows, cols, diag, 1) &&
         !insertrow(A, rows, cols, rows, row, 3) &&
         !insertrow(A, rows, cols, 0, row, cols - 1);
    free2d(A);
    return ok;
}

int main(void)
{
    int fixed[2][3] = {{11, 12, 13}, {21, 22, 23}};
    print_fixed(fixed, 2);

    return demonstrate(2, 3) && demonstrate(3, 2) ? EXIT_SUCCESS : EXIT_FAILURE;
}

**Interpretacja.** Wiersz ma `cols` elementów, a przekątna $\min(\mathrm{rows},\mathrm{cols})$. W macierzy 2×3 drugi wiersz zawiera `21 22 23`, a przekątna `11 22`. W 3×2 wiersz ma długość 2, chociaż macierz ma trzy wiersze. Wynik 0 z funkcji kopiowania lub wstawiania sygnalizuje niepoprawny indeks albo zbyt mały bufor.

## W10.9 — Od pliku do macierzy

Najpierw odczytamy wymiary, potem liczby w kolejności wierszy. Co program powinien zrobić, jeśli w pliku zabraknie ostatniej liczby?

W [samples/data2.dat](samples/data2.dat) najpierw stoją liczby `rows cols`. Dalej jest `rows * cols` liczb całkowitych, w kolejności wierszy. Próbka ma cztery wiersze i trzy kolumny. Uruchom notebook z katalogu wykładu, aby program znalazł podaną ścieżkę względną.

Program sprawdza otwarcie pliku i odczytuje z niego po jednym słowie oddzielonym białymi znakami. Funkcje `strtoumax` i `strtoimax` pozwalają sprawdzić, czy całe słowo jest liczbą i czy mieści się w zakresie. Dopiero poprawną wartość zapisujemy w macierzy. Po ostatniej liczbie plik może zawierać jeszcze białe znaki. Gdy coś się nie powiedzie, program zwalnia pamięć i zamyka plik przed zakończeniem.

In [ ]:
#include <ctype.h>
#include <errno.h>
#include <inttypes.h>
#include <limits.h>
#include <stddef.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>

/* Wczytaj jeden token tekstowy. Odrzuć zbyt długi token w całości. */
static int read_token(FILE *file, char token[64])
{
    size_t length = 0;
    int ch;
    do {
        ch = fgetc(file);
    } while (ch != EOF && isspace(ch));

    while (ch != EOF && !isspace(ch)) {
        if (length == 63 || ch == '\0')
            return 0;
        token[length++] = (char)ch;
        ch = fgetc(file);
    }
    token[length] = '\0';
    return length > 0 && !ferror(file);
}
/* Sprawdzamy cały zapis liczby przed konwersją do size_t. */
static size_t read_size(const char *text, size_t limit)
{
    if (*text < '0' || *text > '9')
        return 0;

    char *end;
    errno = 0;
    uintmax_t value = strtoumax(text, &end, 10);
    if (errno == ERANGE || *end != '\0' || value == 0 || value > limit)
        return 0;
    return (size_t)value;
}
/* Wymiary prostokąta i iloczyny mieszczą się w limitach. */
static int valid_shape(size_t rows, size_t cols)
{
    return rows > 0 && cols > 0 &&
           rows <= SIZE_MAX / sizeof(int *) &&
           rows <= SIZE_MAX / sizeof(int) / cols &&
           rows <= 1000 / cols;
}

static int **alloc2d(size_t rows, size_t cols)
{
    if (!valid_shape(rows, cols))
        return NULL;

    int **A = malloc(rows * sizeof *A);
    if (A == NULL)
        return NULL;

    int *p = malloc(rows * cols * sizeof *p);
    if (p == NULL) {
        free(A);
        return NULL;
    }
    for (size_t i = 0; i < rows; ++i)
        A[i] = p + i * cols;
    return A;
}

static void free2d(int **A)
{
    if (A != NULL) {
        free(A[0]);
        free(A);
    }
}

static void print2d(int **A, size_t rows, size_t cols)
{
    for (size_t i = 0; i < rows; ++i) {
        for (size_t j = 0; j < cols; ++j)
            printf("%d%c", A[i][j], j + 1 == cols ? '\n' : ' ');
    }
}

static int read_value(FILE *file, int *value)
{
    char token[64], *end;
    if (!read_token(file, token))
        return 0;

    errno = 0;
    intmax_t number = strtoimax(token, &end, 10);
    if (end == token || *end != '\0' || errno == ERANGE ||
        number < INT_MIN || number > INT_MAX)
        return 0;
    *value = (int)number;
    return 1;
}

int main(void)
{
    FILE *file = fopen("samples/data2.dat", "r");
    if (file == NULL) {
        fputs("Nie można otworzyć samples/data2.dat.\n", stderr);
        return EXIT_FAILURE;
    }

    char token[64];
    int status = EXIT_FAILURE;
    int **A = NULL;
    size_t rows = read_token(file, token) ? read_size(token, 1000) : 0;
    size_t cols = read_token(file, token) ? read_size(token, 1000) : 0;
    A = alloc2d(rows, cols);
    if (A == NULL)
        goto cleanup;

    for (size_t i = 0; i < rows; ++i)
        for (size_t j = 0; j < cols; ++j)
            if (!read_value(file, &A[i][j]))
                goto cleanup;

    int trailing;
    do {
        trailing = fgetc(file);
    } while (trailing != EOF && isspace(trailing));
    if (trailing != EOF || ferror(file))
        goto cleanup;

    printf("%zu x %zu\n", rows, cols);
    print2d(A, rows, cols);
    if (rows > 1) {
        ptrdiff_t distance = A[1] - A[0];
        printf("A[0] = %p, A[1] = %p\n", (void *)A[0], (void *)A[1]);
        printf("sizeof(int) = %zu\n", sizeof(int));
        printf("Odstęp wierszy = %td elementów, %zu bajtów\n",
               distance, (size_t)distance * sizeof(int));
    }
    status = EXIT_SUCCESS;

cleanup:
    free2d(A);
    if (fclose(file) != 0)
        status = EXIT_FAILURE;
    if (status != EXIT_SUCCESS)
        fputs("Nie udało się wczytać macierzy.\n", stderr);
    return status;
}

**Interpretacja.** Program drukuje cztery wiersze. Adresy pokazane przez `%p` mogą się zmieniać. Odejmowanie `A[1] - A[0]` jest tu dozwolone, bo oba wskaźniki należą do **tego samego bloku** liczb. Wynik to 3 elementy, czyli $3\cdot \mathrm{sizeof}(\mathrm{int})$ bajtów. Przy osobnych przydziałach wierszy takiego odejmowania nie wykonujemy.

## W10.10 — Od gwiazdek do choinki

Za pomocą zagnieżdżonych pętli narysujemy choinkę z gwiazdek i bombek. Gdy zamiast od razu drukować zapiszemy znaki w siatce, będziemy mogli przestawić bombkę przed pokazaniem całego rysunku. Jaką szerokość musi mieć siatka?

### Pętla drukująca

W wersji wyrównanej do lewej w wierszu $i$ drukujemy $i+1$ gwiazdek. W wyśrodkowanej piramidzie o $h$ wierszach dajemy najpierw $h-1-i$ spacji, a potem $2i+1$ znaków. Zaczynamy od $i=0$. Czasem zamiast `*` pojawia się `o`. Stałe ziarno ułatwia porównanie wyników na tej samej platformie.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    enum { HEIGHT = 7 };
    srand(10);

    for (int i = 0; i < HEIGHT; ++i) {
        for (int j = 0; j < HEIGHT - 1 - i; ++j)
            putchar(' ');
        for (int j = 0; j < 2 * i + 1; ++j)
            putchar(rand() % 10 == 0 ? 'o' : '*');
        putchar('\n');
    }
    return 0;
}

### Kilka segmentów i wspólna oś

Można wydzielić rysowanie w funkcji i wywoływać ją dla coraz wyższych segmentów. Wszystkie wywołania dostają tę samą pozycję osi. W dolnym wierszu dodamy pień, a do listy ozdób dodatkowy znak `X`.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

static void draw_triangle(int height, int center)
{
    for (int i = 0; i < height; ++i) {
        for (int j = 0; j < center - i; ++j)
            putchar(' ');
        for (int j = 0; j < 2 * i + 1; ++j) {
            int ornament = rand() % 20;
            putchar(ornament == 0 ? 'X' : ornament < 3 ? 'o' : '*');
        }
        putchar('\n');
    }
}

int main(void)
{
    enum { HEIGHT = 6, CENTER = HEIGHT - 1 };
    srand(10);

    for (int height = 2; height <= HEIGHT; height += 2)
        draw_triangle(height, CENTER);

    for (int j = 0; j < CENTER - 1; ++j)
        putchar(' ');
    puts("|||");
    return 0;
}

### Ćwiczenie. Najpierw zbuduj siatkę, potem ją pokaż

Zastąp drukowanie znaków zapisem do dynamicznego bloku `char`. Użyj $h$ wierszy i $2h-1$ kolumn. Wypełnij tło spacjami i oblicz, w których kolumnach danego wiersza leżą znaki choinki. Przed wydrukiem spróbuj zamienić miejscami `o` i `*` z tego samego wiersza. Wiersze są tu ciągami pól o ustalonej długości. Drukujemy je znak po znaku przez `putchar`.

In [ ]:
//% args: 7
#include <errno.h>
#include <inttypes.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>

/* Sprawdzamy cały zapis liczby przed konwersją do size_t. */
static size_t read_size(const char *text, size_t limit)
{
    if (*text < '0' || *text > '9')
        return 0;

    char *end;
    errno = 0;
    uintmax_t value = strtoumax(text, &end, 10);
    if (errno == ERANGE || *end != '\0' || value == 0 || value > limit)
        return 0;
    return (size_t)value;
}

int main(int argc, char *argv[])
{
    size_t rows = argc == 2 ? read_size(argv[1], 40) : 0;
    if (rows == 0 || rows > SIZE_MAX / 2)
        return EXIT_FAILURE;

    size_t cols = 2 * rows - 1;
    char *grid;
    if (rows > SIZE_MAX / cols / sizeof *grid)
        return EXIT_FAILURE;
    grid = malloc(rows * cols * sizeof *grid);
    if (grid == NULL)
        return EXIT_FAILURE;

    srand(10);
    for (size_t i = 0; i < rows; ++i) {
        size_t first = rows - 1 - i;
        size_t end = rows + i;
        for (size_t j = 0; j < cols; ++j) {
            char pixel = ' ';
            if (first <= j && j < end)
                pixel = rand() % 10 == 0 ? 'o' : '*';
            grid[i * cols + j] = pixel;
        }
    }

    for (size_t i = 0; i < rows; ++i) {
        for (size_t j = 0; j < cols; ++j)
            putchar(grid[i * cols + j]);
        putchar('\n');
    }

    free(grid);
    return EXIT_SUCCESS;
}

**Interpretacja.** Po pominięciu spacji za ostatnim znakiem rysunek zgadza się z wersją z pierwszej komórki. Układ współrzędnych siatki pozwala dotrzeć do konkretnej bombki przez `grid[i * cols + j]` po zbudowaniu rysunku.

**Sprawdź się.**

1. W obu układach macierzy druk działa przez `A[i][j]`. Które adresy przekazujesz do `free` w każdym układzie?
2. Jak z pary `(i, j)` uzyskać pozycję w bloku danych i kiedy ta arytmetyka opisuje pamięć?
3. Co trzeba zwolnić, jeśli przydział trzeciego wiersza się nie udał, a dwa pierwsze się udały?